# Linear Probe: Janus SSM Layers

Probes each layer of the sequence model (Janus SSM) to measure what
information is linearly accessible at each stage of the forward pass.

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, roc_auc_score, r2_score, root_mean_squared_error
import matplotlib.pyplot as plt

from qprimer_designer.models import load_models, PcrDataset
from qprimer_designer.utils import encode_batch_parallel

## 1. Load data and models

In [ ]:
FEAT_RENAME = {
    "f_length": "len_f", "f_Tm": "Tm_f", "f_GC": "GC_f",
    "f_indel": "indel_f", "f_mm": "mm_f",
    "r_length": "len_r", "r_Tm": "Tm_r", "r_GC": "GC_r",
    "r_indel": "indel_r", "r_mm": "mm_r",
    "prod_length": "prod_len",
}
SEQ_RENAME = {
    "f_penc": "pseq_f", "f_tenc": "tseq_f",
    "r_penc": "pseq_r", "r_tenc": "tseq_r",
}
FEATURE_COLUMNS = [
    "len_f", "Tm_f", "GC_f", "indel_f", "mm_f",
    "len_r", "Tm_r", "GC_r", "indel_r", "mm_r",
    "prod_len", "prod_Tm",
]
SEQUENCE_COLUMNS = ["pseq_f", "tseq_f", "pseq_r", "tseq_r"]

DATA_DIR = "."
BATCH_SIZE = 512

rename = {**FEAT_RENAME, **SEQ_RENAME}
train_df = pd.read_csv(f"{DATA_DIR}/0716_dataset_train.csv").rename(columns=rename)
test_df  = pd.read_csv(f"{DATA_DIR}/0716_dataset_test.csv").rename(columns=rename)
print(f"Train: {len(train_df)}, Test: {len(test_df)}")
train_df.head()

In [ ]:
scaler, classifier, regressor, device = load_models(device="cpu")
classifier.eval()
print(f"Device: {device}")
print(classifier)

## 2. Build dataloaders

In [ ]:
def make_loader(df, scaler, batch_size=BATCH_SIZE):
    features = scaler.transform(df[FEATURE_COLUMNS])
    sequences = encode_batch_parallel(df[SEQUENCE_COLUMNS])
    scores = df["score"].values
    return DataLoader(PcrDataset(sequences, features, scores), batch_size=batch_size, shuffle=False)

train_loader = make_loader(train_df, scaler)
test_loader  = make_loader(test_df, scaler)

## 3. Extract representations from every SSM layer

In [ ]:
LAYERS = {
    "encoder": "ssm.encoder",
    "pgc1":    "ssm.pgc1",
    "pgc2":    "ssm.pgc2",
    "norm":    "ssm.norm",
    "s4d":     "ssm.s4d",
    "decoder": "ssm.decoder",
}


def get_submodule(model, dotted_name):
    mod = model
    for p in dotted_name.split("."):
        mod = getattr(mod, p) if not p.isdigit() else mod[int(p)]
    return mod


def extract(model, loader, layer_map):
    activations = {}
    hooks = []

    def make_hook(name):
        def hook_fn(module, inp, out):
            activations.setdefault(name, []).append(out.detach().cpu())
        return hook_fn

    for name, path in layer_map.items():
        h = get_submodule(model, path).register_forward_hook(make_hook(name))
        hooks.append(h)

    all_labels = []
    with torch.no_grad():
        for seq_in, feat_in, labels in loader:
            _ = model(feat_in.float(), seq_in.float())
            all_labels.append(labels.numpy())

    for h in hooks:
        h.remove()

    result = {}
    for name, tensors in activations.items():
        combined = torch.cat(tensors, dim=0)
        if combined.ndim == 3:
            combined = combined.mean(dim=1)
        result[name] = combined.numpy()
    return result, np.concatenate(all_labels)


print("Extracting SSM representations...")
repr_train, y_train = extract(classifier, train_loader, LAYERS)
repr_test,  y_test  = extract(classifier, test_loader,  LAYERS)

y_train_bin = (y_train > 0).astype(int)
y_test_bin  = (y_test > 0).astype(int)

for name, arr in repr_train.items():
    print(f"  {name:12s} -> {arr.shape}")

## 4. Classification probes (score > 0)

In [ ]:
cls_results = {}
for name in LAYERS:
    probe = LogisticRegression(max_iter=2000, solver="lbfgs")
    probe.fit(repr_train[name], y_train_bin)
    y_pred = probe.predict(repr_test[name])
    y_prob = probe.predict_proba(repr_test[name])[:, 1]
    cls_results[name] = {
        "accuracy": accuracy_score(y_test_bin, y_pred),
        "auroc": roc_auc_score(y_test_bin, y_prob),
        "dim": repr_train[name].shape[1],
    }

print(f"{'Layer':<14s} {'Dim':>5s} {'Accuracy':>10s} {'AUROC':>8s}")
print("-" * 40)
for name, m in cls_results.items():
    print(f"  {name:<12s} {m['dim']:>5d} {m['accuracy']:>10.4f} {m['auroc']:>8.4f}")

## 5. Regression probes (raw score)

In [ ]:
reg_results = {}
for name in LAYERS:
    probe = Ridge(alpha=1.0)
    probe.fit(repr_train[name], y_train)
    y_pred = probe.predict(repr_test[name])
    reg_results[name] = {
        "r2": r2_score(y_test, y_pred),
        "rmse": root_mean_squared_error(y_test, y_pred),
        "dim": repr_train[name].shape[1],
    }

print(f"{'Layer':<14s} {'Dim':>5s} {'R²':>10s} {'RMSE':>8s}")
print("-" * 40)
for name, m in reg_results.items():
    print(f"  {name:<12s} {m['dim']:>5d} {m['r2']:>10.4f} {m['rmse']:>8.4f}")

## 6. Visualize

In [ ]:
layer_names = list(LAYERS.keys())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
aurocs = [cls_results[n]["auroc"] for n in layer_names]
bars = ax.barh(layer_names, aurocs)
ax.set_xlabel("AUROC")
ax.set_title("Classification Probe (score > 0)")
ax.set_xlim(0.5, 1.0)
for bar, v in zip(bars, aurocs):
    ax.text(v + 0.005, bar.get_y() + bar.get_height() / 2, f"{v:.3f}", va="center")

ax = axes[1]
r2s = [reg_results[n]["r2"] for n in layer_names]
bars = ax.barh(layer_names, r2s)
ax.set_xlabel("R²")
ax.set_title("Regression Probe (raw score)")
for bar, v in zip(bars, r2s):
    ax.text(max(v + 0.01, 0.01), bar.get_y() + bar.get_height() / 2, f"{v:.3f}", va="center")

plt.tight_layout()
plt.show()